# Time series models (non-DL baselines)

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import time

np.random.seed(42)

In [2]:
DATA_DIR = "../data/"
HORIZON = 28
SEASON = 7

In [3]:
sales = pd.read_csv(DATA_DIR + "sales_train_evaluation.csv")
day_cols = [c for c in sales.columns if c.startswith("d_")]

In [4]:
# build 1 time series model per item for the top store CA_3
sales_ca3 = sales[sales["store_id"] == "CA_3"].reset_index(drop=True)
series_matrix = sales_ca3[day_cols].values.astype(np.float32)
ids = sales_ca3["id"].values

In [5]:
series_matrix
# 3049 items, 1941 days 

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 2., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 5., 4., 0.]], shape=(3049, 1941), dtype=float32)

In [6]:
# keep the last 28 days as test data
y_train = series_matrix[:, :-HORIZON]
y_test = series_matrix[:, -HORIZON:]

# Seasonal Naive

In [7]:
# Model 1: seasonal naive
last_week = y_train[:, -SEASON:]
# just repeat the last 7 days 4 times  
y_pred = np.tile(last_week, (HORIZON // SEASON + 1))[:, :HORIZON]

In [8]:
forecast_cols = [f"F{i}" for i in range(1, HORIZON + 1)]
results_naive = pd.DataFrame(y_pred, columns=forecast_cols)
# append item ID
results_naive.insert(0, "id", ids) 

In [9]:
# evaluate model 1
rmse_naive = np.sqrt(((y_pred - y_test) ** 2).mean())
print(f"Seasonal Naive RMSE: {rmse_naive:.2f}")
print(results_naive.head())
# results.to_csv("seasonal_naive_forecasts.csv", index=False)

Seasonal Naive RMSE: 3.32
                              id    F1   F2   F3   F4   F5    F6   F7    F8  \
0  HOBBIES_1_001_CA_3_evaluation   0.0  1.0  1.0  1.0  0.0   3.0  3.0   0.0   
1  HOBBIES_1_002_CA_3_evaluation   2.0  0.0  0.0  0.0  0.0   1.0  0.0   2.0   
2  HOBBIES_1_003_CA_3_evaluation   0.0  0.0  0.0  0.0  0.0   1.0  1.0   0.0   
3  HOBBIES_1_004_CA_3_evaluation  15.0  3.0  0.0  3.0  4.0  12.0  3.0  15.0   
4  HOBBIES_1_005_CA_3_evaluation   1.0  2.0  0.0  0.0  0.0   5.0  1.0   1.0   

    F9  ...  F19   F20  F21   F22  F23  F24  F25  F26   F27  F28  
0  1.0  ...  0.0   3.0  3.0   0.0  1.0  1.0  1.0  0.0   3.0  3.0  
1  0.0  ...  0.0   1.0  0.0   2.0  0.0  0.0  0.0  0.0   1.0  0.0  
2  0.0  ...  0.0   1.0  1.0   0.0  0.0  0.0  0.0  0.0   1.0  1.0  
3  3.0  ...  4.0  12.0  3.0  15.0  3.0  0.0  3.0  4.0  12.0  3.0  
4  2.0  ...  0.0   5.0  1.0   1.0  2.0  0.0  0.0  0.0   5.0  1.0  

[5 rows x 29 columns]


# SARIMA

In [10]:
# Model 2: SARIMA
sales_small = sales_ca3.sample(30)
results = []
rmses = []
fallback_count = 0

t0 = time.time()

# 1 SARIMA model per item 

for i, row in sales_small.iterrows():
    train = row[day_cols].values[:-HORIZON].astype(float)
    actual = row[day_cols].values[-HORIZON:].astype(float)
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("error")
            model = SARIMAX(train, order=(0,1,0), seasonal_order=(0,1,0,7),
                            enforce_stationarity=False, enforce_invertibility=False)
            pred = model.fit(disp=False, maxiter=200).forecast(steps=HORIZON)
            pred = np.clip(pred, 0, None)
    except Exception:
        pred = np.tile(train[-7:], HORIZON // 7 + 1)[:HORIZON]
        fallback_count += 1

    rmses.append(np.sqrt(((pred - actual) ** 2).mean()))
    results.append([row["id"]] + list(pred))
    print(f"{i}/{len(sales)} | RMSE: {np.mean(rmses):.2f} | Fallbacks: {fallback_count}")

df = pd.DataFrame(results, columns=["id"] + forecast_cols)
df.to_csv("sarima_forecasts_CA3.csv", index=False)
print(f"\nFinal RMSE: {np.mean(rmses):.2f} | Fallbacks: {fallback_count}/{len(sales_small)}")

t1 = time.time()
print(f"\nSARIMA modeling completed in {(t1 - t0) / 60:.2f} minutes.")

1517/30490 | RMSE: 0.53 | Fallbacks: 1
2369/30490 | RMSE: 0.70 | Fallbacks: 2
1961/30490 | RMSE: 1.45 | Fallbacks: 3
343/30490 | RMSE: 1.41 | Fallbacks: 4
2663/30490 | RMSE: 1.58 | Fallbacks: 5
1173/30490 | RMSE: 1.43 | Fallbacks: 6
2676/30490 | RMSE: 1.43 | Fallbacks: 7
1027/30490 | RMSE: 1.33 | Fallbacks: 8
2083/30490 | RMSE: 1.59 | Fallbacks: 9
1444/30490 | RMSE: 1.56 | Fallbacks: 10
932/30490 | RMSE: 1.52 | Fallbacks: 11
178/30490 | RMSE: 1.43 | Fallbacks: 12
1593/30490 | RMSE: 1.46 | Fallbacks: 13
1947/30490 | RMSE: 1.50 | Fallbacks: 14
1662/30490 | RMSE: 1.57 | Fallbacks: 15
2431/30490 | RMSE: 1.73 | Fallbacks: 16
2861/30490 | RMSE: 1.70 | Fallbacks: 17
2342/30490 | RMSE: 1.81 | Fallbacks: 18
1497/30490 | RMSE: 1.79 | Fallbacks: 19
2965/30490 | RMSE: 1.98 | Fallbacks: 20
2413/30490 | RMSE: 1.94 | Fallbacks: 21
2807/30490 | RMSE: 2.03 | Fallbacks: 22
1283/30490 | RMSE: 2.05 | Fallbacks: 23
2867/30490 | RMSE: 2.07 | Fallbacks: 24
889/30490 | RMSE: 2.06 | Fallbacks: 25
1459/30490 | 

In [12]:
# df_sarima = pd.DataFrame(results, columns=["id"] + forecast_cols)
# df.to_csv("sarima_forecasts_CA3.csv", index=False)
# print(f"\nFinal Mean RMSE (CA_3): {np.mean(rmses):.2f}")

Issues:
- Many sparse series so SARIMA failed to converge. SARIMA is unsuitable for this data

# End